# Task 1: Inference Only
## Model: Google Gemma 4 31B 

This notebook performs automated answering of 644 questions regarding Austrian tax law using the **Google Gemma 4 31B** model via the Google AI Studio API.

**Approach:**
* **Model:** Gemma 4 31B.
* **System Prompting:** We instruct the model to act as an expert in Austrian Tax Law (EStG, KStG, BAO).
* **Output:** A CSV file containing the question `id` and the model's `answer`.

### 1. Libraries and Environment Setup
First, we install and import the necessary libraries. We use python-dotenv to securely load the API key from a .env file.

In [1]:
# Install required libraries (run once if not already installed)
# !pip install -U google-genai pandas

import os
import time
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_fixed

# Load environment variables from .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("System Error: GOOGLE_API_KEY not found. Check your .env file.")

# Initialize the Gemini Client
client = genai.Client(api_key=api_key)

print("Libraries imported and client initialized successfully.")

Libraries imported and client initialized successfully.


### 2. Configuration and Paths
We define the model and file paths.

In [2]:
# Model name of Google Gemma 4 31B
MODEL_ID = "gemma-4-31b-it" 

# Relative paths: dataset is in Team1/, code is in Team1/code/
INPUT_FILE = "../dataset_clean.csv"
OUTPUT_FOLDER = "../results/"
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "inference_gemma_results.csv")

# Ensure the results directory exists
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created directory: {OUTPUT_FOLDER}")

### 3. Define Inference Function
This function sends a prompt to the Gemini model with specific system instructions to act as an Austrian tax law expert.

In [3]:
@retry(wait=wait_fixed(5), stop=stop_after_attempt(5))
def get_tax_expert_answer(prompt: str) -> str:
    """
    Sends a prompt to Gemini with system instructions for Austrian tax law.
    Includes a retry mechanism for rate limits (Error 429).
    """
    system_instruction = (
        "You are a highly qualified expert in Austrian tax law. "
        "Provide precise, professional answers. Always reference "
        "relevant paragraphs (e.g., EStG, KStG, BAO, UStG) where applicable. "
        "The answer must be in German and formatted as one continuous paragraph."
    )
    
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.1,  # Low temperature for high factual accuracy
            max_output_tokens=500
        )
    )
    return response.text.strip()
    
    

### 4. Main Execution Loop
We load the dataset (644 rows) and iterate through each question, saving progress periodically.

In [ ]:
# Load the dataset (expected 644 rows)
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Successfully loaded {len(df)} questions from {INPUT_FILE}")
except FileNotFoundError:
    print(f"Error: {INPUT_FILE} not found. Please check your file paths.")
    raise

# Initialize an empty list to store results
results = []

print(f"Starting inference... Estimated completion time: ~2 hours.")
print(f"Note: Respecting 5 RPM limit (13s delay per request).")

# Iterate through each row in the dataset
for index, row in df.iterrows():
    question_id = row['id']
    user_prompt = row['prompt']
    
    # Log progress to the console so you can see the process is active
    print(f"[{index + 1}/{len(df)}] Processing ID: {question_id}...", end=" ", flush=True)
    
    # Request answer from the tax expert function (Tenacity handles 429/503 errors)
    answer = get_tax_expert_answer(user_prompt)
    
    # Format the answer into a single continuous paragraph for CSV compatibility
    clean_answer = " ".join(answer.split())
    
    # Append only the required 'id' and 'answer' columns
    results.append({
        "id": question_id,
        "answer": clean_answer
    })
    
    print("Done.")

    # Intermediate save every 5 rows to ensure no data is lost during the long run
    if (index + 1) % 5 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        print(f"--- Progress saved to {OUTPUT_FILE} ---")
    
    # CRITICAL: Mandatory 5-second sleep to strictly follow the 5 RPM Free Tier limit
    time.sleep(5)

# Final save to capture the remaining rows and ensure correct formatting
final_df = pd.DataFrame(results)
final_df = final_df[['id', 'answer']] # Ensure strict column order
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nInference completed successfully!")
print(f"Final results saved to: {OUTPUT_FILE}")

Successfully loaded 643 questions from ../dataset_clean.csv
Starting inference... Estimated completion time: ~2 hours.
Note: Respecting 5 RPM limit (13s delay per request).
[1/643] Processing ID: CORP-TAX-001... 

Done.
[2/643] Processing ID: CORP-TAX-002... Done.
[3/643] Processing ID: CORP-TAX-003... Done.
[4/643] Processing ID: CORP-TAX-004... Done.
[5/643] Processing ID: CORP-TAX-005... Done.
--- Progress saved to ../results/inference_gemma_results.csv ---
[6/643] Processing ID: CORP-TAX-006... Done.
[7/643] Processing ID: CORP-TAX-007... Done.
[8/643] Processing ID: CORP-TAX-008... Done.
[9/643] Processing ID: CORP-TAX-009... Done.
[10/643] Processing ID: CORP-TAX-010... Done.
--- Progress saved to ../results/inference_gemma_results.csv ---
[11/643] Processing ID: CORP-TAX-011... Done.
[12/643] Processing ID: CORP-TAX-012... Done.
[13/643] Processing ID: CORP-TAX-013... Done.
[14/643] Processing ID: CORP-TAX-014... Done.
[15/643] Processing ID: CORP-TAX-015... Done.
--- Progress saved to ../results/inference_gemma_results.csv ---
[16/643] Processing ID: CORP-TAX-016... Done.
[17/643] Processing ID: CORP-TAX-017... Done.
[18/643] Processing ID: CORP-TAX-018... Done.
[19/643] Processing ID: C

RetryError: RetryError[<Future at 0x118534980 state=finished raised ServerError>]

In [ ]:
df = pd.read_csv(INPUT_FILE)

# Read the existing file so we don't start from scratch
if os.path.exists(OUTPUT_FILE):
    existing_df = pd.read_csv(OUTPUT_FILE)
    # Collect IDs that have already been processed
    processed_ids = set(existing_df['id'].astype(str))
    # Convert the existing DataFrame back to a list of dictionaries
    results = existing_df.to_dict('records')
    print(f"Resuming... Found {len(processed_ids)} already processed questions.")
else:
    processed_ids = set()
    results = []
    print("Starting from scratch...")

print(f"Target: {len(df)} questions. Safe limit: 15 RPM (4.5s delay).")

# Main Loop
for index, row in df.iterrows():
    question_id = str(row['id'])
    
    # IF THE QUESTION IS ALREADY PROCESSED -> SKIP IT
    if question_id in processed_ids:
        continue

    print(f"[{index + 1}/{len(df)}] Processing {question_id}...", end=" ", flush=True)
    
    # 3. Error Catching (prevents script crash on ServerError)
    try:
        answer = get_tax_expert_answer(row['prompt'])
        clean_answer = " ".join(answer.split())
    except Exception as e:
        print(f" FAILED. Skipping...")
        clean_answer = "Error: Server failed"
    
    # Add the new result
    results.append({"id": question_id, "answer": clean_answer})
    processed_ids.add(question_id)
    print("Done.")

    # Save progress every 5 questions
    # Calculate how many NEW rows we've processed in this run
    newly_processed = len(results) - len(existing_df) if 'existing_df' in locals() else len(results)
    
    if newly_processed > 0 and newly_processed % 5 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        print(f"--- Saved progress (Total: {len(results)} rows) ---")
    
    # Safe delay for Gemma 4 31B (15 RPM limit)
    time.sleep(4.5) 

# Final Save to ensure strict column formatting
final_df = pd.DataFrame(results)[['id', 'answer']]
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nInference completed successfully!")
print(f"Results available at: {OUTPUT_FILE}")

Resuming... Found 260 already processed questions.
Target: 643 questions. Safe limit: 15 RPM (4.5s delay).
[261/643] Processing GRESt-AT-021... Done.
[262/643] Processing GRESt-AT-022... Done.
[263/643] Processing GRESt-AT-023... Done.
[264/643] Processing GRESt-AT-024... Done.
[265/643] Processing GRESt-AT-025... Done.
--- Saved progress (Total: 265 rows) ---
[266/643] Processing GRESt-AT-026... Done.
[267/643] Processing GRESt-AT-027... Done.
[268/643] Processing GRESt-AT-028... Done.
[269/643] Processing GRESt-AT-029... Done.
[270/643] Processing GRESt-AT-030... Done.
--- Saved progress (Total: 270 rows) ---
[271/643] Processing GRESt-AT-031... Done.
[272/643] Processing GRESt-AT-032... Done.
[273/643] Processing GRESt-AT-033... Done.
[274/643] Processing GRESt-AT-034... Done.
[275/643] Processing GRESt-AT-035... Done.
--- Saved progress (Total: 275 rows) ---
[276/643] Processing GRESt-AT-036... Done.
[277/643] Processing GRESt-AT-037... Done.
[278/643] Processing GRESt-AT-038... Do